# Khởi tạo Cơ sở Dữ liệu Vector cho RAG với BGE-Large-En-v1.5

Thực hiện việc chuyển đổi tập dữ liệu tri thức `data/processed/rag_knowledge_base.csv` thành cơ sở dữ liệu vector ChromaDB phục vụ cho RAG.

**Cải tiến mô hình Embedding:**
- Chúng ta sử dụng mô hình **`BAAI/bge-large-en-v1.5`**.
- Mô hình mở rộng giới hạn ngữ cảnh **512 tokens**, giúp bao trọn toàn bộ các bài luận IELTS Writing Task 2 mà không bị cắt cụt ngữ nghĩa và tăng độ chính xác tìm kiếm tương đồng.

**Cấu hình Cache (Anti-Trash C-Drive):**
- Sử dụng thư viện `python-dotenv` để nạp tệp cấu hình `.env` chuyển đổi toàn bộ thư mục tải mô hình (Hugging Face Cache, Unsloth Cache) sang ổ `T:` giúp bảo vệ ổ `C:` khỏi bị tràn bộ nhớ.

**Nguyên tắc chống rò rỉ dữ liệu (Anti-Leakage Principle):**
- Chỉ sử dụng tệp `rag_knowledge_base.csv` (vốn được nhân bản từ tập Train 90% sau khi đã xử lý leakage ở Notebook 2 và 3).
- Tuyệt đối không đưa dữ liệu từ tập Validation (`val.csv`) hoặc tập Test (`test.csv`) vào Vector Database này.

In [ ]:
import os
from dotenv import load_dotenv

# Nạp cấu hình môi trường (.env) trước khi import bất kỳ thư viện nào của Hugging Face
load_dotenv(os.path.abspath("../.env"))

import pandas as pd
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

## 1. Thiết lập Đường dẫn & Đọc dữ liệu tri thức

In [ ]:
DATA_DIR = "../data/processed"
KB_FILE = os.path.join(DATA_DIR, "rag_knowledge_base.csv")
VECTOR_DB_DIR = os.path.join(DATA_DIR, "chroma_db")

df_kb = pd.read_csv(KB_FILE)
print(f"Đã đọc {len(df_kb)} dòng tri thức từ: {KB_FILE}")

## 2. Tạo đối tượng LangChain Document

Kết hợp `Prompt` đề bài và `Essay` thành nội dung văn bản tìm kiếm (`page_content`), đồng thời đưa điểm Overall Band cùng điểm các tiêu chí thành phần vào trường `metadata` để RAG truy xuất chính xác.

In [ ]:
documents = []

for index, row in df_kb.iterrows():
    # Định dạng nội dung lưu trữ trong vector
    page_content = (
        f"Prompt: {row['prompt']}\n\n"
        f"Essay: {row['essay']}"
    )
    
    # Metadata chứa điểm số chi tiết
    metadata = {
        "Overall_Band": float(row['Overall_Band']),
        "TR_Band": float(row['TR_Band']),
        "CC_Band": float(row['CC_Band']),
        "LR_Band": float(row['LR_Band']),
        "GRA_Band": float(row['GRA_Band'])
    }
    
    documents.append(Document(page_content=page_content, metadata=metadata))

print(f"Đã tạo thành công {len(documents)} đối tượng LangChain Document.")

## 3. Khởi tạo Mô hình Embedding mới & Tạo ChromaDB cục bộ

Sử dụng mô hình `BAAI/bge-large-en-v1.5` chạy trên GPU (cuda).

In [ ]:
model_name = "BAAI/bge-large-en-v1.5"
model_kwargs = {"device": "cuda"} # Sử dụng GPU RTX 4060

print(f"Đang tải mô hình Embedding mới: {model_name}...")
embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)

print(f"Đang mã hóa văn bản và lưu cơ sở dữ liệu vector tại: {VECTOR_DB_DIR}...")
vectordb = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=VECTOR_DB_DIR
)

print("✔ Hoàn thành tạo Vector Database mới với BGE-Large-En-v1.5!")